# Implement Vision Transformer + MAE Pretraining

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google, Apple, Tesla, Waymo

---

### Problem Statement

The **Vision Transformer (ViT)** treats images as sequences of patches and processes them with a standard Transformer encoder. **Masked Autoencoder (MAE)** pretraining randomly masks a high percentage (75%) of patches, feeds only the visible patches through the encoder, and uses a lightweight decoder to reconstruct the masked patches. This forces the encoder to learn meaningful visual representations.

Your task is to implement a ViT with MAE pretraining from scratch. You will work with small synthetic images to keep training tractable.

---

### Requirements

1. **PatchEmbedding** — Split an image into non-overlapping patches and project each patch to an embedding vector.
2. **ViT** — Patch embedding + [CLS] token + learnable positional embedding + Transformer encoder layers.
3. **MAE** — Random masking of 75% of patches, encoder processes only visible patches, lightweight decoder reconstructs masked patches from encoder output + mask tokens.
4. **Validation** — Reconstruction loss should decrease. Show that encoder learns by comparing patch representations before/after training.

---

### Constraints

- ✅ Use PyTorch for all implementations.
- ✅ Use small images (32x32) with small patches (8x8) for tractability.
- ✅ Use synthetic data (random patterns or simple shapes).
- ✅ The decoder can be much smaller than the encoder (1-2 layers).
- ❌ Do **not** use `torchvision.models` or any pre-built ViT implementations.

---

<details>
  <summary>💡 Hint</summary>

  - For PatchEmbedding, use `nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)` as an efficient way to extract and project patches simultaneously.
  - In MAE, the encoder only sees ~25% of patches. Append mask tokens (learned embedding) at the masked positions before feeding to the decoder.
  - The decoder input needs positional embeddings for ALL positions (both visible and masked) so it knows where each token belongs spatially.
  - Reconstruction loss is MSE between predicted and original pixel values for masked patches only.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generate synthetic image data: simple patterns
torch.manual_seed(42)
np.random.seed(42)

def generate_synthetic_images(n_samples, img_size=32, channels=3):
    """
    Generate synthetic images with simple patterns (gradients, circles, stripes)
    to give the model something structured to learn.
    """
    images = torch.zeros(n_samples, channels, img_size, img_size)
    y = torch.linspace(-1, 1, img_size)
    x = torch.linspace(-1, 1, img_size)
    yy, xx = torch.meshgrid(y, x, indexing='ij')

    for i in range(n_samples):
        pattern = i % 4
        if pattern == 0:  # horizontal gradient
            for c in range(channels):
                freq = np.random.uniform(1, 3)
                phase = np.random.uniform(0, 2 * np.pi)
                images[i, c] = torch.sin(freq * xx + phase)
        elif pattern == 1:  # vertical gradient
            for c in range(channels):
                freq = np.random.uniform(1, 3)
                phase = np.random.uniform(0, 2 * np.pi)
                images[i, c] = torch.sin(freq * yy + phase)
        elif pattern == 2:  # circular pattern
            cx, cy = np.random.uniform(-0.3, 0.3, 2)
            r = torch.sqrt((xx - cx)**2 + (yy - cy)**2)
            for c in range(channels):
                freq = np.random.uniform(2, 5)
                images[i, c] = torch.sin(freq * r)
        else:  # checkerboard
            freq = np.random.choice([2, 4, 8])
            checker = torch.sign(torch.sin(freq * np.pi * xx) * torch.sin(freq * np.pi * yy))
            for c in range(channels):
                images[i, c] = checker * np.random.choice([-1, 1])

    # Normalize to [0, 1]
    images = (images - images.min()) / (images.max() - images.min() + 1e-8)
    return images


img_size = 32
patch_size = 8
channels = 3
n_train = 2000

train_images = generate_synthetic_images(n_train, img_size, channels)
print(f'Training images shape: {train_images.shape}')

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(train_images[i].permute(1, 2, 0).numpy())
    axes[i].set_title(f'Sample {i}')
    axes[i].axis('off')
plt.suptitle('Synthetic Training Images')
plt.tight_layout()
plt.show()

In [ ]:
# --- Part 1: Patch Embedding ---

class PatchEmbedding(nn.Module):
    """
    Split image into non-overlapping patches and project to embedding dimension.

    For a 32x32 image with 8x8 patches: 16 patches total.
    Each patch has patch_size * patch_size * channels = 8*8*3 = 192 values.
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        # TODO: Use Conv2d with kernel_size=patch_size, stride=patch_size
        # This efficiently extracts and projects all patches in one operation
        # self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        ...

    def forward(self, x):
        """
        Args:
            x (Tensor): Images, shape (B, C, H, W).

        Returns:
            Tensor: Patch embeddings, shape (B, n_patches, embed_dim).
        """
        # TODO: Apply Conv2d projection, flatten spatial dims, transpose
        # proj output: (B, embed_dim, H/P, W/P) -> flatten to (B, embed_dim, n_patches) -> transpose to (B, n_patches, embed_dim)
        ...


# Test
pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=128)
test_img = torch.randn(2, 3, 32, 32)
test_patches = pe(test_img)
print(f'PatchEmbedding: {test_img.shape} -> {test_patches.shape}')
print(f'Number of patches: {pe.n_patches}')

In [ ]:
# --- Part 2: Vision Transformer ---

class ViT(nn.Module):
    """
    Vision Transformer: patch embedding + [CLS] token + positional embedding + transformer encoder.
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128,
                 n_heads=4, n_layers=4, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches

        # TODO: Create [CLS] token (learnable, shape 1, 1, embed_dim)
        # self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        # TODO: Create positional embeddings for CLS + all patches
        # self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)

        # TODO: Create Transformer encoder layers
        # encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads,
        #                                            dim_feedforward=int(embed_dim * mlp_ratio),
        #                                            batch_first=True, norm_first=True)
        # self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # TODO: Layer norm
        # self.norm = nn.LayerNorm(embed_dim)
        ...

    def forward(self, x, return_all_tokens=False):
        """
        Args:
            x (Tensor): Images, shape (B, C, H, W).
            return_all_tokens (bool): If True, return all tokens; otherwise return CLS only.

        Returns:
            Tensor: [CLS] token representation (B, embed_dim) or all tokens (B, 1+n_patches, embed_dim).
        """
        # TODO: Get patch embeddings
        # TODO: Prepend [CLS] token
        # TODO: Add positional embeddings
        # TODO: Pass through transformer encoder
        # TODO: Apply layer norm
        # TODO: Return CLS token or all tokens
        ...


vit = ViT(img_size=32, patch_size=8, in_channels=3, embed_dim=128, n_heads=4, n_layers=4)
test_out = vit(test_img)
print(f'ViT output (CLS only): {test_out.shape}')
test_all = vit(test_img, return_all_tokens=True)
print(f'ViT output (all tokens): {test_all.shape}')

In [ ]:
# --- Part 3: Masked Autoencoder (MAE) ---

class MAE(nn.Module):
    """
    Masked Autoencoder for self-supervised ViT pretraining.

    Key idea:
    - Randomly mask 75% of patches
    - Encoder processes only visible (25%) patches
    - Lightweight decoder reconstructs all patches from encoder output + mask tokens
    - Loss is MSE on masked patches only
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3,
                 encoder_embed_dim=128, encoder_heads=4, encoder_layers=4,
                 decoder_embed_dim=64, decoder_heads=4, decoder_layers=1,
                 mask_ratio=0.75):
        super().__init__()
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio
        n_patches = (img_size // patch_size) ** 2
        self.n_patches = n_patches
        patch_dim = patch_size * patch_size * in_channels

        # TODO: Encoder (patch embedding + positional embedding + transformer)
        # self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, encoder_embed_dim)
        # self.encoder_pos_embed = nn.Parameter(torch.randn(1, n_patches, encoder_embed_dim) * 0.02)
        # encoder_layer = nn.TransformerEncoderLayer(...)
        # self.encoder = nn.TransformerEncoder(...)
        # self.encoder_norm = nn.LayerNorm(encoder_embed_dim)

        # TODO: Decoder
        # self.decoder_embed = nn.Linear(encoder_embed_dim, decoder_embed_dim)
        # self.mask_token = nn.Parameter(torch.randn(1, 1, decoder_embed_dim) * 0.02)
        # self.decoder_pos_embed = nn.Parameter(torch.randn(1, n_patches, decoder_embed_dim) * 0.02)
        # decoder_layer = nn.TransformerEncoderLayer(...)
        # self.decoder = nn.TransformerEncoder(...)
        # self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        # self.decoder_pred = nn.Linear(decoder_embed_dim, patch_dim)  # predict pixel values
        ...

    def random_masking(self, x):
        """
        Randomly mask patches.

        Args:
            x (Tensor): Patch embeddings, shape (B, N, D).

        Returns:
            x_visible (Tensor): Visible patches only, shape (B, N_visible, D).
            mask (Tensor): Binary mask, shape (B, N). 1 = masked, 0 = visible.
            ids_restore (Tensor): Indices to restore original order, shape (B, N).
        """
        # TODO: Generate random noise per patch, sort to get random permutation
        # TODO: Keep first (1 - mask_ratio) patches as visible
        # TODO: Return visible patches, binary mask, and restore indices
        ...

    def forward(self, images):
        """
        Args:
            images (Tensor): Input images, shape (B, C, H, W).

        Returns:
            loss (Tensor): MSE reconstruction loss on masked patches.
            pred (Tensor): Predicted patches, shape (B, N, patch_dim).
            mask (Tensor): Binary mask showing which patches were masked.
        """
        # TODO: Step 1 - Get patch embeddings and add positional embeddings
        # TODO: Step 2 - Apply random masking (encoder sees only visible patches)
        # TODO: Step 3 - Encode visible patches
        # TODO: Step 4 - Prepare decoder input: project encoder output + insert mask tokens
        # TODO: Step 5 - Add decoder positional embeddings and decode
        # TODO: Step 6 - Predict pixel values for all patches
        # TODO: Step 7 - Compute MSE loss on masked patches only
        ...


mae = MAE(img_size=32, patch_size=8, in_channels=3,
          encoder_embed_dim=128, encoder_heads=4, encoder_layers=4,
          decoder_embed_dim=64, decoder_heads=4, decoder_layers=1)
print(f'MAE parameters: {sum(p.numel() for p in mae.parameters()):,}')

# Test forward pass
test_loss, test_pred, test_mask = mae(train_images[:4])
print(f'Test loss: {test_loss.item():.4f}')
print(f'Predictions shape: {test_pred.shape}')
print(f'Mask shape: {test_mask.shape}, masked ratio: {test_mask.float().mean():.2f}')

In [ ]:
# --- Part 4: Training ---

optimizer = torch.optim.AdamW(mae.parameters(), lr=1e-3, weight_decay=0.05)
n_epochs = 100
batch_size = 64
losses = []

mae.train()
for epoch in range(n_epochs):
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_train, batch_size):
        imgs = train_images[perm[i:i+batch_size]]

        loss, pred, mask = mae(imgs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('MAE Training Loss')
plt.show()

In [ ]:
# --- Validation ---

mae.eval()

def unpatchify(pred, patch_size=8, img_size=32, channels=3):
    """Convert patch predictions back to image format."""
    h = w = img_size // patch_size
    pred = pred.reshape(-1, h, w, patch_size, patch_size, channels)
    pred = pred.permute(0, 5, 1, 3, 2, 4)  # B, C, h, p, w, p
    pred = pred.reshape(-1, channels, img_size, img_size)
    return pred


def patchify(imgs, patch_size=8):
    """Convert images to patches."""
    B, C, H, W = imgs.shape
    h = w = H // patch_size
    patches = imgs.reshape(B, C, h, patch_size, w, patch_size)
    patches = patches.permute(0, 2, 4, 3, 5, 1)  # B, h, w, p, p, C
    patches = patches.reshape(B, h * w, -1)  # B, N, patch_dim
    return patches


with torch.no_grad():
    test_imgs = train_images[:4]
    loss, pred, mask = mae(test_imgs)

    # Reconstruct images
    reconstructed = unpatchify(pred, patch_size, img_size, channels)

    # Show original, masked, and reconstructed
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for i in range(4):
        # Original
        axes[0, i].imshow(test_imgs[i].permute(1, 2, 0).numpy().clip(0, 1))
        axes[0, i].set_title('Original')
        axes[0, i].axis('off')

        # Masked (show only visible patches)
        masked_img = test_imgs[i].clone()
        patch_mask = mask[i].reshape(img_size // patch_size, img_size // patch_size)
        for pi in range(img_size // patch_size):
            for pj in range(img_size // patch_size):
                if patch_mask[pi, pj] == 1:  # masked
                    masked_img[:, pi*patch_size:(pi+1)*patch_size,
                               pj*patch_size:(pj+1)*patch_size] = 0.5
        axes[1, i].imshow(masked_img.permute(1, 2, 0).numpy().clip(0, 1))
        axes[1, i].set_title('Masked Input')
        axes[1, i].axis('off')

        # Reconstructed
        axes[2, i].imshow(reconstructed[i].permute(1, 2, 0).numpy().clip(0, 1))
        axes[2, i].set_title('Reconstructed')
        axes[2, i].axis('off')

    plt.suptitle('MAE Reconstruction Results')
    plt.tight_layout()
    plt.show()

# Checks
print('=== Validation ===')
print(f'Final training loss: {losses[-1]:.6f}')
assert losses[-1] < losses[0], 'Reconstruction loss should decrease!'
print('PASSED: Loss decreased during training')

# Check encoder produces different representations for different patterns
with torch.no_grad():
    patch_emb = mae.patch_embed(test_imgs)
    pos_emb = patch_emb + mae.encoder_pos_embed
    encoded = mae.encoder(pos_emb)
    encoded = mae.encoder_norm(encoded)

    # Cosine similarity between first two images' mean representations
    repr_0 = encoded[0].mean(0)
    repr_1 = encoded[1].mean(0)
    repr_2 = encoded[2].mean(0)

    sim_01 = F.cosine_similarity(repr_0.unsqueeze(0), repr_1.unsqueeze(0)).item()
    sim_02 = F.cosine_similarity(repr_0.unsqueeze(0), repr_2.unsqueeze(0)).item()
    print(f'Cosine similarity (img0 vs img1): {sim_01:.4f}')
    print(f'Cosine similarity (img0 vs img2): {sim_02:.4f}')
    print('PASSED: Encoder produces varied representations')

print('\nAll validations passed!')